In [104]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from jax import grad, vmap, random
import optax
import diffrax
from seaborn import kdeplot
import wandb
from ot.lp import wasserstein_1d as wd
import argparse
import pickle

from triangular_transport.flows.flow_trainer import (
    NNTrainer,
)

from triangular_transport.flows.interpolants import (
    linear_interpolant,
    linear_interpolant_der,
    trig_interpolant,
    trig_interpolant_der,
    sigmoid_interpolant,
    sigmoid_interpolant_der,
    linear_interpolant_noise,
    linear_interpolant_der_noise
)
from triangular_transport.flows.loss_functions import vec_field_loss
from triangular_transport.networks.flow_networks import MLP
from triangular_transport.flows.methods.sampling import inf_train_gen
from triangular_transport.flows.dataloaders import (
    standard_gaussian_reference_sampler,
    gaussian_reference_sampler
)
plt.style.use("ggplot")

In [3]:
nsamples = 100000
seed = 1
rng = np.random.RandomState(seed)
samps0 = np.load("rej_samples_0.npy")
samps1 = np.load("rej_samples_1.npy")
samps4 = np.load("rej_samples_4.npy")
us_base = rng.randn(nsamples, 1) * 2
base_wd0 = wasserstein_1d(
    us_base,
    samps0,
    p=2,
)
base_wd1 = wasserstein_1d(
    us_base,
    samps1,
    p=2,
)
base_wd4 = wasserstein_1d(
    us_base,
    samps4,
    p=2,
)

In [141]:
cond_no0 = 0.0
cond_no1 = -1.0
cond_no4 = -4.2
sample_no = 20000
cond_vals = [cond_no0, cond_no1, cond_no4]
epochs = 400
rng2 = np.random.RandomState(0)
x1_data = inf_train_gen(data="banana", rng=rng2, batch_size=100000)

key = random.PRNGKey(14 + 1 + 0)
key, key1, key2, key3, key4 = random.split(key, 5)
key, subkey1 = random.split(key, 2)
train_dim = 20000
batch_size = 2048
# batch_size = hyperparams["batch_size"]
batch_size = min(batch_size, train_dim)
batch_size -= 1
steps_per_epoch = int(np.ceil(train_dim / batch_size))
steps = steps_per_epoch * epochs
print_every = 10000
yu_dimension = (1, 1)
dim = yu_dimension[0] + yu_dimension[1]
# hidden_layer_list = [256] * 4 if train_dim < 8000 else [1024] * 8
# hidden_layer_list = [512] * 6
hidden_layer_list = [256] * 3
# hidden_layer_list = [1024] * 8
target_data = x1_data[:sample_no, :]
model = MLP(
    key=key1,
    dim=dim,
    time_varying=True,
    w=hidden_layer_list,
    num_layers=len(hidden_layer_list) + 1,
    activation_fn=jax.nn.gelu,  # GeLU worked well
)
schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=1e-3,
    warmup_steps=2000,
    decay_steps=steps,
    end_value=1e-5,
)

optimizer = optax.chain(
    optax.clip_by_global_norm(1.0), optax.adamw(schedule)
)
# optimizer = optax.adam(1e-5)

interpolant = linear_interpolant
interpolant_der = linear_interpolant_der
interpolant_args = {"t": None, "x1": None, "x0": None}
reference_sampler_args = {"mu": 0, "sigma": 2.0}

In [142]:
trainer = NNTrainer(
    target_density=None,
    model=model,
    optimizer=optimizer,
    interpolant=interpolant,
    interpolant_der=interpolant_der,
    reference_sampler=gaussian_reference_sampler,
    loss=vec_field_loss,
    interpolant_args=interpolant_args,
    yu_dimension=yu_dimension,
    reference_sampler_args=reference_sampler_args,
)

trainer.train(
    train_data=target_data,
    train_dim=train_dim,
    batch_size=batch_size,
    steps=steps,
    x0_data=None,
    print_every=print_every,
);

Training neural network


  6%|▌         | 236/4000 [00:01<00:12, 309.61it/s]

step = 0, train_loss = -0.007786022499203682


100%|██████████| 4000/4000 [00:04<00:00, 905.35it/s] 

step = 3999, train_loss = -2.49337100982666


In [143]:
solver_args = {"solver": diffrax.Dopri5(), "max_steps": 50000, "stepsize_controller": diffrax.PIDController(rtol=1e-4, atol=1e-6)}
cond_samples = trainer.conditional_sample(
    cond_values=cond_vals,
    nsamples=nsamples,
    u0_cond=None,
    solver_args=solver_args
)

In [144]:
wd(np.array(cond_samples[0][:, 1]), samps0.squeeze(), p=2) / base_wd0

array([0.00128911])

In [145]:
wd(np.array(cond_samples[1][:, 1]), samps1.squeeze(), p=2) / base_wd1

array([0.00809817])

In [146]:
wd(np.array(cond_samples[2][:, 1]), samps4.squeeze(), p=2) / base_wd4

array([0.13252787])